# AR(p) synthetic series vs TimesFM 2.5 (Parallel)

Run Monte Carlo replications in parallel across CPU processes, then aggregate summary statistics centrally.

## 1. Setup

This notebook reuses `ar_vs_timesfm_worker.py`, which runs each `(rep, p)` task in a process pool and returns per-task metrics.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
load_dotenv(REPO / ".env")

if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from ar_vs_timesfm_worker import run_monte_carlo_parallel

In [ ]:
N_REPLICATIONS = 8
N = 720
K_FIRST = 360

# Keep this modest: each worker loads its own TimesFM model instance.
MAX_WORKERS = min(4, os.cpu_count() or 4)

# Base seed used to derive deterministic per-task seeds in workers.
RNG_SEED = 20260410

VERBOSE_PROGRESS = True
PROGRESS_EVERY_TASKS = 1

## 2. Run Parallel Monte Carlo

In [ ]:
results = run_monte_carlo_parallel(
    rng_seed=RNG_SEED,
    n_replications=N_REPLICATIONS,
    n=N,
    k_first=K_FIRST,
    repo_root=REPO,
    max_workers=MAX_WORKERS,
    verbose_progress=VERBOSE_PROGRESS,
    progress_every_tasks=PROGRESS_EVERY_TASKS,
)

summary = (
    results.groupby("p_dgp", as_index=False)
    .agg(
        n=("rep", "count"),
        mse_ar_mean=("mse_ar", "mean"),
        mse_ar_std=("mse_ar", "std"),
        mse_timesfm_mean=("mse_timesfm", "mean"),
        mse_timesfm_std=("mse_timesfm", "std"),
        dm_pvalue_median=("dm_pvalue_two_sided", "median"),
        dm_reject_rate_5pct=("dm_pvalue_two_sided", lambda s: float(np.mean(s < 0.05))),
        ar_fit_failures_total=("ar_fit_failures", "sum"),
    )
)

overall = pd.DataFrame(
    {
        "n_tasks": [len(results)],
        "n_replications": [N_REPLICATIONS],
        "max_workers": [MAX_WORKERS],
        "mse_ar_overall": [float(results["mse_ar"].mean())],
        "mse_timesfm_overall": [float(results["mse_timesfm"].mean())],
        "dm_pvalue_median_overall": [float(results["dm_pvalue_two_sided"].median())],
    }
)

display(summary)
display(overall)
display(results.head(8))

# AR(p) synthetic series vs TimesFM 2.5 (Parallel)

Run Monte Carlo replications in parallel across CPU processes, then compute summary statistics centrally.

## 1. Setup

This notebook reuses `ar_vs_timesfm_worker.py`, which loads TimesFM once per worker process and executes each `(rep, p)` task in parallel.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
load_dotenv(REPO / ".env")

if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from ar_vs_timesfm_worker import run_monte_carlo_parallel

In [ ]:
N_REPLICATIONS = 8
N = 720
K_FIRST = 360

# Keep this modest: each worker has its own TimesFM model instance.
MAX_WORKERS = min(4, os.cpu_count() or 4)

# Base seed used to derive per-task seeds in workers.
RNG_SEED = 20260410

VERBOSE_PROGRESS = True
PROGRESS_EVERY_TASKS = 1

## 2. Run Parallel Monte Carlo

In [ ]:
results = run_monte_carlo_parallel(
    rng_seed=RNG_SEED,
    n_replications=N_REPLICATIONS,
    n=N,
    k_first=K_FIRST,
    repo_root=REPO,
    max_workers=MAX_WORKERS,
    verbose_progress=VERBOSE_PROGRESS,
    progress_every_tasks=PROGRESS_EVERY_TASKS,
)

summary = (
    results.groupby("p_dgp", as_index=False)
    .agg(
        n=("rep", "count"),
        mse_ar_mean=("mse_ar", "mean"),
        mse_ar_std=("mse_ar", "std"),
        mse_timesfm_mean=("mse_timesfm", "mean"),
        mse_timesfm_std=("mse_timesfm", "std"),
        dm_pvalue_median=("dm_pvalue_two_sided", "median"),
        dm_reject_rate_5pct=("dm_pvalue_two_sided", lambda s: float(np.mean(s < 0.05))),
        ar_fit_failures_total=("ar_fit_failures", "sum"),
    )
)

overall = pd.DataFrame(
    {
        "n_tasks": [len(results)],
        "n_replications": [N_REPLICATIONS],
        "max_workers": [MAX_WORKERS],
        "mse_ar_overall": [float(results["mse_ar"].mean())],
        "mse_timesfm_overall": [float(results["mse_timesfm"].mean())],
        "dm_pvalue_median_overall": [float(results["dm_pvalue_two_sided"].median())],
    }
)

display(summary)
display(overall)
display(results.head(8))

## 3. Save Outputs

In [ ]:
out_dir = HERE / "output"
out_dir.mkdir(parents=True, exist_ok=True)

path_rep = out_dir / "ar_vs_timesfm_parallel_replications.csv"
path_sum = out_dir / "ar_vs_timesfm_parallel_summary.csv"
path_overall = out_dir / "ar_vs_timesfm_parallel_overall.csv"

results.to_csv(path_rep, index=False)
summary.to_csv(path_sum, index=False)
overall.to_csv(path_overall, index=False)

print(path_rep)
print(path_sum)
print(path_overall)

# AR(p) synthetic series vs TimesFM 2.5 (Parallel)

Run Monte Carlo replications in parallel across CPU processes, then aggregate all statistics centrally.

## 1. Setup

This notebook uses `ar_vs_timesfm_worker.py` to execute `(rep, p)` tasks in a `ProcessPoolExecutor`.
Each worker process loads TimesFM once, runs one task, and returns metrics to the parent process.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
load_dotenv(REPO / ".env")

if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from ar_vs_timesfm_worker import run_monte_carlo_parallel

In [ ]:
N_REPLICATIONS = 8
N = 720
K_FIRST = 360

# Number of worker processes for parallel tasks.
# Keep this modest because each worker loads its own TimesFM model copy.
MAX_WORKERS = min(4, os.cpu_count() or 4)

# Base seed used to derive deterministic per-task seeds.
RNG_SEED = 20260410

VERBOSE_PROGRESS = True
PROGRESS_EVERY_TASKS = 1

## 2. Run Parallel Monte Carlo

In [ ]:
results = run_monte_carlo_parallel(
    rng_seed=RNG_SEED,
    n_replications=N_REPLICATIONS,
    n=N,
    k_first=K_FIRST,
    repo_root=REPO,
    max_workers=MAX_WORKERS,
    verbose_progress=VERBOSE_PROGRESS,
    progress_every_tasks=PROGRESS_EVERY_TASKS,
)

summary = (
    results.groupby("p_dgp", as_index=False)
    .agg(
        n=("rep", "count"),
        mse_ar_mean=("mse_ar", "mean"),
        mse_ar_std=("mse_ar", "std"),
        mse_timesfm_mean=("mse_timesfm", "mean"),
        mse_timesfm_std=("mse_timesfm", "std"),
        dm_pvalue_median=("dm_pvalue_two_sided", "median"),
        dm_reject_rate_5pct=("dm_pvalue_two_sided", lambda s: float(np.mean(s < 0.05))),
        ar_fit_failures_total=("ar_fit_failures", "sum"),
    )
)

overall = pd.DataFrame(
    {
        "n_tasks": [len(results)],
        "n_replications": [N_REPLICATIONS],
        "max_workers": [MAX_WORKERS],
        "mse_ar_overall": [float(results["mse_ar"].mean())],
        "mse_timesfm_overall": [float(results["mse_timesfm"].mean())],
        "dm_pvalue_median_overall": [float(results["dm_pvalue_two_sided"].median())],
    }
)

display(summary)
display(overall)
display(results.head(8))

## 3. Save Outputs